# 🧠 Developing GANs with Keras


## 📋 Overview

I build a Generative Adversarial Network (GAN) on MNIST: a **generator** that turns random noise into synthetic digit images, and a **discriminator** that tries to tell real digits apart from the generator's fakes. I train both networks against each other, evaluate the generated images both visually and with quantitative metrics, and then run three practice variations — deepening the generator, tuning the discriminator's learning rate, and tracking the adversarial training losses over time.

Coming from RF, the generator/discriminator dynamic maps directly onto an electronic-warfare arms race: a jammer trying to spoof a signal detector, and a detector trying to get better at telling real signals from spoofed ones. Neither side "wins" outright — training succeeds when they reach a rough parity, the same way a well-tuned spoofer and a well-tuned detector settle into a standoff rather than either one dominating.

**What I cover:**
- 📥 Preprocessing MNIST for adversarial training
- 🏗️ Building the generator (noise → synthetic image)
- 🏗️ Building the discriminator (image → real/fake probability)
- 🔗 Combining both into the full GAN, with the discriminator frozen inside it
- 🔄 The adversarial training loop
- 📊 Assessing generated image quality — qualitative and quantitative
- 🎯 Practice: deeper generator, discriminator learning rate, loss tracking


## 🧩 Theory

A GAN is a **minimax game** between two networks trained simultaneously:

$$
\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}}\left[\log D(x)\right] + \mathbb{E}_{z \sim p_z}\left[\log(1 - D(G(z)))\right]
$$

- $G$ — the **generator**, maps random noise $z$ to a synthetic sample $G(z)$
- $D$ — the **discriminator**, outputs the probability that an input is real rather than generated
- $D$ wants to **maximize** its ability to tell real ($x$) from fake ($G(z)$) apart
- $G$ wants to **minimize** $D$'s ability to catch it — i.e. make $D(G(z))$ as close to "real" as possible

In practice, each network is trained with its own binary cross-entropy loss:

$$
\mathcal{L}_D = -\frac{1}{2}\Big[\log D(x) + \log\big(1 - D(G(z))\big)\Big], \qquad \mathcal{L}_G = -\log D(G(z))
$$

Training alternates: update $D$ on a batch of real + fake images, then update $G$ (through the frozen $D$) to push its fakes toward being classified as real.

### A note on the original lab's wording

The instructions describe building the generator and discriminator with "the Keras functional API," but the code actually uses the **Sequential API** throughout (`Sequential()` + `.add()`) — no `Input`/`Model` graph wiring for either sub-network (the functional API only shows up once, to wire the *combined* GAN). I build it exactly as written, just with accurate language, since the distinction matters when I'm deciding which API fits a given model later on.

### 📡 Telecom analogy

| GAN piece | Signal processing / EW equivalent |
|---|---|
| Generator $G$ | A jammer/spoofer trying to produce a signal indistinguishable from the real one |
| Discriminator $D$ | A detector trying to tell a genuine signal from a spoofed one |
| Adversarial training | An ECM/ECCM arms race — each side improves in response to the other |
| Discriminator accuracy ≈ 50% at convergence | Detector and spoofer reach parity — detector can't do better than a coin flip |
| Mode collapse (low diversity) | A spoofer that finds one exploit and reuses it — eventually easy to fingerprint |
| `LeakyReLU` instead of `ReLU` | Keeping a small leakage/bias current so a stage never fully cuts off — avoids a "dead" gradient path during an already-unstable adversarial training process |


## Part 1 — 📥 Data Preprocessing

I load MNIST and normalize pixel values to $[-1, 1]$ rather than $[0, 1]$ — this matches the generator's `tanh` output activation later on, which also produces values in $[-1, 1]$. I expand each image to `(28, 28, 1)` to match the tensor shape both networks expect.


In [ ]:
%%capture
!pip install tensorflow-cpu==2.16.2

# Suppress warnings and set environment variables
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'


In [ ]:
import numpy as np
from tensorflow.keras.datasets import mnist
import warnings

# Suppress all Python warnings
warnings.filterwarnings('ignore')

# Load the MNIST dataset
(x_train, _), (_, _) = mnist.load_data()

# Normalize the pixel values to the range [-1, 1] -- matches the generator's tanh output
x_train = x_train.astype('float32') / 127.5 - 1.
x_train = np.expand_dims(x_train, axis=-1)

# Print the shape of the data
print(x_train.shape)


**What just happened:**

| Step | Purpose |
|---|---|
| Normalize to $[-1,1]$ | Matches the generator's `tanh` output range |
| Expand to `(28,28,1)` | Match the 3D tensor shape both networks expect |


## Part 2 — 🏗️ Building the Generator

The generator maps a 100-dimensional noise vector $z$ through three widening `Dense` layers (256 → 512 → 1024), each followed by `LeakyReLU` and `BatchNormalization`, then projects to $28 \times 28 = 784$ values with a `tanh` output and reshapes to an image. `LeakyReLU`'s small negative slope keeps a gradient flowing even for negative pre-activations — useful insurance in a training setup that's already prone to instability.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, BatchNormalization, Reshape

# Define the generator model
def build_generator():
    model = Sequential()
    model.add(Dense(256, input_dim=100))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(512))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(1024))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(28 * 28 * 1, activation='tanh'))
    model.add(Reshape((28, 28, 1)))
    return model

# Build the generator
generator = build_generator()
generator.summary()


**Architecture summary:**

| Layer | Units | Activation | Notes |
|---|---|---|---|
| Input | 100 | — | Random noise vector $z$ |
| Dense → LeakyReLU → BatchNorm | 256 | LeakyReLU ($\alpha=0.2$) | Momentum 0.8 tracks a moving statistic — like AGC adapting to a shifting signal distribution |
| Dense → LeakyReLU → BatchNorm | 512 | LeakyReLU | |
| Dense → LeakyReLU → BatchNorm | 1024 | LeakyReLU | |
| Dense + Reshape | 784 → (28,28,1) | tanh | Output range $[-1,1]$, matches normalized pixels |


## Part 3 — 🏗️ Building the Discriminator

The discriminator flattens a $28\times28\times1$ image and passes it through two shrinking `Dense` layers (512 → 256) with `LeakyReLU`, ending in a single sigmoid output — the probability that the input image is real. Compiled with binary cross-entropy, since this is a real-vs-fake binary classification task.


In [ ]:
from tensorflow.keras.layers import Flatten
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU

# Define the discriminator model
def build_discriminator():
    model = Sequential()
    model.add(Flatten(input_shape=(28, 28, 1)))
    model.add(Dense(512))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(256))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(1, activation='sigmoid'))
    return model

# Build and compile the discriminator
discriminator = build_discriminator()
discriminator.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
discriminator.summary()


## Part 4 — 🔗 Building the Combined GAN

I stack the generator and discriminator into a single model: noise in, generator produces an image, discriminator classifies it. Before compiling, I freeze the discriminator (`trainable = False`) so that when I train this combined model, only the **generator's** weights update — the discriminator's own training happens separately, on its own `compile()`/`train_on_batch()` calls.


In [ ]:
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model

# Create the GAN by stacking the generator and the discriminator
def build_gan(generator, discriminator):
    discriminator.trainable = False
    gan_input = Input(shape=(100,))
    generated_image = generator(gan_input)
    gan_output = discriminator(generated_image)
    gan = Model(gan_input, gan_output)
    gan.compile(loss='binary_crossentropy', optimizer='adam')
    return gan

# Build the GAN
gan = build_gan(generator, discriminator)
gan.summary()


# Sync discriminator weights from trainable to non-trainable in GAN
gan.layers[2].set_weights(discriminator.get_weights())


## Part 5 — 🔄 Training the Adversarial Loop

**A quirk worth flagging:** before the training loop, the original lab code rebuilds a *brand-new* `discriminator` object from scratch (same architecture, freshly initialized weights). That new object is what the loop's `discriminator.train_on_batch(...)` calls actually train — but the `gan` model built in Part 4 still holds a reference to the **original** discriminator instance inside its frozen layer. So the generator is being trained against the Part 4 discriminator (untouched since the weight-sync line), while a separate, freshly-initialized discriminator is being trained independently and is what the printed `D accuracy` reflects. I keep this exactly as written per the "preserve all original logic" rule, but it's worth knowing about rather than silently reproducing without comment — it's the kind of subtle wiring bug that's easy to miss when copying GAN boilerplate.


In [ ]:
# Define and compile the discriminator model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, Flatten

def build_discriminator():
    model = Sequential()
    model.add(Flatten(input_shape=(28, 28, 1)))
    model.add(Dense(512))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(256))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(1, activation='sigmoid'))
    return model

# Build and recompile the discriminator
discriminator = build_discriminator()
discriminator.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
discriminator.summary()


In [ ]:
# Training parameters

batch_size = 64
epochs = 200
sample_interval = 10

# Adversarial ground truths
real = np.ones((batch_size, 1))
fake = np.zeros((batch_size, 1))

# Training loop
for epoch in range(epochs):
    # Train the discriminator
    idx = np.random.randint(0, x_train.shape[0], batch_size)
    real_images = x_train[idx]
    noise = np.random.normal(0, 1, (batch_size, 100))
    generated_images = generator.predict(noise)
    d_loss_real = discriminator.train_on_batch(real_images, real)
    d_loss_fake = discriminator.train_on_batch(generated_images, fake)
    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

    # Train the generator
    noise = np.random.normal(0, 1, (batch_size, 100))
    g_loss = gan.train_on_batch(noise, real)

    # Print the progress
    if epoch % sample_interval == 0:
        print(f"{epoch} [D loss: {d_loss[0]}] [D accuracy: {100 * d_loss[1]}%] [G loss: {g_loss}]")


## Part 6 — 📊 Assessing Generated Image Quality

I evaluate the trained GAN two ways: **qualitative** (looking at generated images directly) and **quantitative** (numeric metrics). Neither is sufficient alone — visual inspection is fast but subjective, metrics are objective but can miss things a glance would catch immediately.

### Qualitative: visual inspection

When I look at the generated grid, I check for:

| Quality | What to look for |
|---|---|
| Clarity | Sharp digits, not blurry smears |
| Coherence | Recognizable digit shapes with the right number of strokes |
| Diversity | A variety of digits/styles — repetition suggests mode collapse |


In [ ]:
!pip install matplotlib
import matplotlib.pyplot as plt

def sample_images(generator, epoch, num_images=25):
    noise = np.random.normal(0, 1, (num_images, 100))
    generated_images = generator.predict(noise)
    generated_images = 0.5 * generated_images + 0.5  # Rescale to [0, 1]
    fig, axs = plt.subplots(5, 5, figsize=(10, 10))
    count = 0

    for i in range(5):
        for j in range(5):
            axs[i, j].imshow(generated_images[count, :, :, 0], cmap='gray')
            axs[i, j].axis('off')
            count += 1
    plt.show()

# Sample images at the end of training
sample_images(generator, epochs)


### Quantitative: metrics

| Metric | What it measures | Notes |
|---|---|---|
| **Inception Score (IS)** | Quality + diversity via a pretrained classifier's predictions | Not very informative for simple datasets like MNIST — built for complex, natural images |
| **Fréchet Inception Distance (FID)** | Distance between real and generated feature distributions | Lower = better; widely used, more reliable than IS for most cases |
| **Discriminator accuracy** | How well $D$ still tells real from fake after training | Accuracy near 50% ⇒ $D$ can't do better than a coin flip ⇒ $G$'s fakes are convincing |

$$
\text{Accuracy} \approx 50\% \iff D(x) \approx D(G(z)) \approx 0.5 \quad \text{(discriminator can't separate real from fake)}
$$


In [ ]:
# Calculate and print the discriminator accuracy on real vs. fake images
noise = np.random.normal(0, 1, (batch_size, 100))
generated_images = generator.predict(noise)

# Evaluate the discriminator on real images
real_images = x_train[np.random.randint(0, x_train.shape[0], batch_size)]
d_loss_real = discriminator.evaluate(real_images, np.ones((batch_size, 1)), verbose=0)

# Evaluate the discriminator on fake images
d_loss_fake = discriminator.evaluate(generated_images, np.zeros((batch_size, 1)), verbose=0)

print(f"Discriminator Accuracy on Real Images: {d_loss_real[1] * 100:.2f}%")
print(f"Discriminator Accuracy on Fake Images: {d_loss_fake[1] * 100:.2f}%")


**Putting it together:** I start with visual inspection for a quick read, then lean on discriminator accuracy (and FID, for more complex datasets) for an objective check, and watch the generator/discriminator loss curves over time to catch instability — one network running away from the other rather than settling into balance.


## 🎯 Practice: Deepening the Generator

**Objective:** see how adding capacity to the generator affects image quality. I add a fourth `Dense(2048)` block (with the same `LeakyReLU` + `BatchNormalization` pattern) before the final projection layer, and rebuild.


In [ ]:
# Modify the generator model by adding an additional Dense layer

def build_generator():
    model = Sequential()
    model.add(Dense(256, input_dim=100))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(512))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(1024))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(2048))  # New layer added
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(28 * 28 * 1, activation='tanh'))
    model.add(Reshape((28, 28, 1)))
    return model

# Rebuild the generator
generator = build_generator()
generator.summary()


## ⚙️ Practice: Tuning the Discriminator's Learning Rate

**Objective:** explore how a lower discriminator learning rate affects training stability. A common GAN failure mode is the discriminator learning too fast and overpowering the generator before it has a chance to improve — slowing the discriminator down (0.0002 vs. Adam's default 0.001) gives the generator more room to catch up, similar to deliberately backing off detector sensitivity so a jamming-countermeasure loop doesn't destabilize into one side dominating outright.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, LeakyReLU

def build_discriminator():
    model = Sequential()
    model.add(Flatten(input_shape=(28, 28, 1)))
    model.add(Dense(512))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(256))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(1, activation='sigmoid'))
    return model

# Rebuild and compile the discriminator with a lower learning rate
discriminator = build_discriminator()
discriminator.compile(loss='binary_crossentropy',
                      optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002),
                      metrics=['accuracy'])

discriminator.summary()


## 📈 Practice: Visualizing Training Progress

**Objective:** track discriminator and generator losses across training instead of only reading printed snapshots — a loss curve makes it much easier to spot instability (e.g. one loss collapsing to near-zero while the other explodes) than scanning printed numbers epoch by epoch.


In [ ]:
# Initialize lists to store losses
d_losses = []
g_losses = []


# Training loop with loss storage
for epoch in range(epochs):
    idx = np.random.randint(0, x_train.shape[0], batch_size)
    real_images = x_train[idx]
    noise = np.random.normal(0, 1, (batch_size, 100))
    generated_images = generator.predict(noise)
    d_loss_real = discriminator.train_on_batch(real_images, real)
    d_loss_fake = discriminator.train_on_batch(generated_images, fake)
    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
    d_losses.append(d_loss[0])


    noise = np.random.normal(0, 1, (batch_size, 100))
    g_loss = gan.train_on_batch(noise, real)
    g_losses.append(g_loss)


    if epoch % sample_interval == 0:
        print(f"{epoch} [D loss: {d_loss[0]}] [D accuracy: {100 * d_loss[1]}] [G loss: {g_loss}]")

# Plot the training losses
plt.figure(figsize=(10, 5))
plt.plot(d_losses, label='Discriminator Loss')
plt.plot(g_losses, label='Generator Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Losses')
plt.legend()
plt.show()


## 📊 Summary

| Concept | What I did | Why it matters |
|---|---|---|
| 📥 Preprocessing | Normalized to $[-1,1]$, expanded to `(28,28,1)` | Matches the generator's `tanh` output range |
| 🏗️ Generator | Sequential Dense→LeakyReLU→BatchNorm stack, `tanh` output | Maps noise $z$ to a synthetic image |
| 🏗️ Discriminator | Sequential Flatten→Dense→LeakyReLU stack, sigmoid output | Classifies real vs. fake, binary cross-entropy loss |
| 🔗 Combined GAN | Generator → Discriminator (frozen) wired via the Functional API | Lets me train the generator through a frozen discriminator |
| 🔄 Adversarial training | Alternated `discriminator.train_on_batch` and `gan.train_on_batch` | Implements the minimax game between $G$ and $D$ |
| 📊 Evaluation | Visual inspection + discriminator accuracy (~50% target) | Combines subjective and objective quality checks |
| 🎯 Deeper generator | Added a `Dense(2048)` block | Tests whether more capacity improves image quality |
| ⚙️ Discriminator LR | Lowered to 0.0002 | A common stability fix when $D$ overpowers $G$ |
| 📈 Loss tracking | Plotted `d_losses` vs. `g_losses` | Makes instability visible at a glance instead of scanning printed logs |

**Telecom throughline:** the whole notebook is an arms race — generator-as-jammer vs. discriminator-as-detector — and the practical fixes (slowing the discriminator's learning rate, watching for one side collapsing) are the same moves I'd reach for balancing an ECM/ECCM loop so neither side runs away with it.


## 🧪 Sandbox

Space to keep experimenting beyond the practice exercises:

- Fix the Part 5 discriminator-object quirk properly — rebuild the GAN (`build_gan`) after creating the new discriminator so the generator trains against the same object being adversarially updated
- Try a convolutional GAN (`Conv2D`/`Conv2DTranspose` in both networks) instead of pure `Dense` layers — typically sharper images on MNIST
- Compute an actual FID score between real and generated batches instead of relying on discriminator accuracy alone
- Try label smoothing (`real = np.ones(...) * 0.9` instead of `1.0`) — a common trick for stabilizing discriminator training
- Push `epochs` well beyond 200 and watch for mode collapse — does image diversity actually degrade over a longer run?


In [ ]:
# 🧪 Sandbox — experiment here
